In [1]:
import geopandas as gpd
from rasterstats import zonal_stats
import pandas as pd
import rasterio
import os
import numpy as np
from rasterio.mask import mask

In [2]:
base = os.path.join(os.getcwd(),'..')

In [3]:
shapefile_path = os.path.join(os.getcwd(),'..','shape','States_shapefile.shp')

In [4]:
outPath = os.path.join(base,'stats')
if not os.path.exists(outPath):
    os.mkdir(outPath)

In [5]:
def get_zonal_values(rp1,rp2,fo_csv):
    gdf = gpd.read_file(shapefile_path)
    with rasterio.open(rp1) as src1, rasterio.open(rp2) as src2:
        if gdf.crs != src1.crs:
            gdf = gdf.to_crs(src1.crs)
    
        results = []
        band_raster1 = 0
        band_raster2 = 0
        for idx, row in gdf.iterrows():
            geom = [row.geometry]
    
            out1, _ = mask(src1, geom, crop=True)
            band1 = out1[band_raster1 - 1].astype("float32")
    
            out2, _ = mask(src2, geom, crop=True)
            band2 = out2[band_raster2 - 1].astype("float32")
            
            nodata1 = src1.nodata
            nodata2 = src2.nodata
            mask_nodata = np.zeros(band1.shape, dtype=bool)
            if nodata1 is not None:
                mask_nodata |= (band1 == nodata1)
            if nodata2 is not None:
                mask_nodata |= (band2 == nodata2)
            
            band1 = np.where(band1==0,np.nan,1)
            
            mult = band1 * band2
            mult[mask_nodata] = np.nan
    
            stats = {
                "zone_id": idx,
                "mean": np.nanmean(mult),
                "sum": np.nansum(mult),
                "min": np.nanmin(mult),
                "max": np.nanmax(mult),
                "count": np.sum(~np.isnan(mult))
            }
    
            results.append(stats)
    
    df = pd.DataFrame(results)
    df = pd.concat([gdf.reset_index(drop=True), df], axis=1)
    df.drop(columns=['FID', 'Program', 'State_Code','Flowing_St', 'FID_1','geometry', 'zone_id',  'count'],inplace=True)
    df.to_csv(fo_csv, index=False)
    return df[[ 'State_Name', 'mean']]

In [6]:
def get_3(t):
    n = t//12
    x = len(str(n))
    y = '0'*(3-x)+str(n)
    return y

In [8]:
var_name = 'aboveground_biomassc'
cult = 'irrigated'
os.makedirs(os.path.join(outPath,var_name),exist_ok=True)
compiled = pd.DataFrame()
for t in range(0,10):
    sd_str = get_3(t)    
    rp1 = os.path.join(base,'raster(clip)','sdate','sdate_time_'+sd_str+'_'+cult+'.tif')
    rp2 = os.path.join(base,'raster(clip)',var_name,var_name+'_time_'+str(t)+'_'+cult+'.tif')
    # rp2 = os.path.join(base,'raster(clip)',var_name,var_name+'_time_'+sd_str+'_'+cult+'.tif')
    fo_csv = os.path.join(outPath,var_name,var_name+'_'+str(t)+'_'+cult+'.csv')
    df = get_zonal_values(rp1,rp2,fo_csv)
    df.rename(columns={'mean':t},inplace=True)
    if t==0:
        compiled = df
    else:
        compiled = pd.merge(compiled,df,on='State_Name')
        
compiled.to_excel(os.path.join(outPath,var_name,var_name+'_'+cult+'_compiled.xlsx'))